# Vector Databases
### Practice Notebook

**Assumed pre-installed libraries:** `numpy`, `sentence-transformers`,
`faiss-cpu`, `chromadb`


## 1. Why vector DBs are needed

Day 3's `semantic_search` function re-embeds and brute-force compares the
*entire* corpus against every query — fine for few sentences, hopeless for a
knowledge base with a million chunks. A vector database exists to solve two
problems at scale:

1. **Fast approximate nearest-neighbor (ANN) search** — instead of comparing
   a query against every single stored vector (`O(n)`), specialized indexes
   let you find the top-k most similar vectors in far less time, trading a
   small amount of accuracy for large speed gains.
2. **Persistence and management** — storing vectors alongside their
   metadata, supporting inserts/updates/deletes, and filtering search by
   metadata (e.g., "only search chunks from 2025 policy documents").

## 2. Indexing concepts (conceptual level)

- **Flat / brute-force index** — no cleverness, compares the query to every
  vector. Exact results, but slow at scale. Good baseline / ground truth.
- **IVF (Inverted File Index)** — clusters vectors into `nlist` buckets
  (via k-means) at index-build time. At query time, only searches the
  handful of buckets closest to the query, instead of the whole dataset.
  Trade-off: `nprobe` (how many buckets to check) controls the
  speed/accuracy trade-off — check more buckets, get better recall, at
  higher latency.
- **HNSW (Hierarchical Navigable Small World)** — builds a multi-layer graph
  where each vector is linked to its approximate nearest neighbors; search
  "navigates" the graph from a coarse top layer down to a fine-grained
  bottom layer. Generally gives very high recall with fast query times, at
  the cost of higher memory usage and slower index-build time than IVF.

Rule of thumb for class discussion: **HNSW** is the most common default in
modern vector DBs (great recall/speed balance for most workloads up to tens
of millions of vectors); **IVF** (often combined with product quantization,
IVF-PQ) is preferred when memory is tight and the dataset is very large.


## 3. Common vector stores at a glance

| Store | Type | Notes |
|---|---|---|
| **FAISS** (Meta) | Library, not a full DB | Extremely fast ANN search (flat, IVF, HNSW, PQ); no built-in persistence/server, metadata handling, or network API — you build that yourself |
| **Chroma** | Lightweight embedded/self-hosted DB | Easiest to get started with locally; built-in metadata filtering and persistence; good for prototyping and small-to-medium projects |
| **Pinecone** | Managed cloud service | Fully managed, scales automatically, no infra to run yourself; usage-based pricing |
| **Weaviate** | Self-hosted or managed | Full-featured (hybrid search, built-in modules for embeddings), GraphQL/REST API, good for production deployments needing more control |

Let's use FAISS for the indexing concepts (HNSW vs. flat, directly), and
Chroma for the more realistic "store chunks with metadata and query them"
workflow.


## 4. FAISS: flat vs. HNSW index

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

corpus = [
    "The cat sat on the mat.",
    "A dog was running in the park.",
    "Machine learning models can generate text.",
    "The feline rested on the rug.",
    "Stock prices fell sharply after the announcement.",
    "Large language models are a type of machine learning model.",
    "The puppy played fetch in the garden.",
    "Neural networks are trained using backpropagation.",
    "Investors reacted to the earnings report.",
    "The kitten napped on the sofa.",
]

embeddings = model.encode(corpus).astype("float32")
dim = embeddings.shape[1]
print("Embedding dimension:", dim)


In [ ]:
# --- Flat index: exact brute-force search ---
flat_index = faiss.IndexFlatL2(dim)
flat_index.add(embeddings)
print("Vectors in flat index:", flat_index.ntotal)

# --- HNSW index: approximate, graph-based search ---
hnsw_index = faiss.IndexHNSWFlat(dim, 32)   # 32 = neighbors per node (M)
hnsw_index.hnsw.efConstruction = 40
hnsw_index.add(embeddings)
print("Vectors in HNSW index:", hnsw_index.ntotal)


In [ ]:
def search(index, query, k=3):
    query_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, k)
    return [(corpus[i], float(d)) for i, d in zip(indices[0], distances[0])]

query = "A small dog is playing outside."

print("Flat index results:")
for text, dist in search(flat_index, query):
    print(f"  {dist:.3f}  {text}")

print("\nHNSW index results:")
for text, dist in search(hnsw_index, query):
    print(f"  {dist:.3f}  {text}")


On a 10-sentence toy corpus, flat and HNSW should return (near) identical
results — the speed difference only becomes visible at scale (thousands to
millions of vectors). That's expected and is itself the point: HNSW
approximates the flat index's results, faster, as data grows.

**Exercise 4.1:** Increase `corpus` to 50+ sentences (duplicate/paraphrase
the existing ones, or add your own), and use Python's `time` module to
compare `search()` latency between `flat_index` and `hnsw_index`. At what
corpus size (roughly) does HNSW start to noticeably win?


## 5. Chroma: storing and querying embeddings with metadata

Chroma is closer to what you'd actually reach for in a small-to-medium RAG
project — it manages persistence and metadata for you, so you don't have to
build that layer yourself on top of FAISS.


In [ ]:
import chromadb

chroma_client = chromadb.Client()   # in-memory client for this notebook;
                                     # use chromadb.PersistentClient(path=...)
                                     # to persist to disk between sessions
collection = chroma_client.get_or_create_collection(name="rag_practice")

# Reuse the chunk-with-metadata pattern from Day 2
documents = corpus
metadatas = [
    {"topic": "animals"} if i in (0, 1, 3, 6, 9) else
    {"topic": "ai_ml"} if i in (2, 5, 7) else
    {"topic": "finance"}
    for i in range(len(corpus))
]
ids = [f"doc_{i}" for i in range(len(corpus))]

collection.add(documents=documents, metadatas=metadatas, ids=ids)
print("Documents stored:", collection.count())


In [ ]:
results = collection.query(
    query_texts=["A small dog is playing outside."],
    n_results=3,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0],
                            results["distances"][0]):
    print(f"{dist:.3f}  [{meta['topic']}]  {doc}")


In [ ]:
# Metadata filtering: only search within the 'ai_ml' topic
filtered_results = collection.query(
    query_texts=["How are these systems trained?"],
    n_results=2,
    where={"topic": "ai_ml"},
)

for doc, meta, dist in zip(filtered_results["documents"][0],
                            filtered_results["metadatas"][0],
                            filtered_results["distances"][0]):
    print(f"{dist:.3f}  [{meta['topic']}]  {doc}")


**Note:** by default Chroma uses its own built-in embedding function if you
don't supply one. In a real pipeline you'd typically pass `embeddings=`
directly (pre-computed with the *same* model used elsewhere in your
pipeline, e.g., the `sentence-transformers` model above) to guarantee
consistency between indexing and querying.

**Exercise 4.2 (mini deliverable):** Take the chunk records from Day 2 /
Day 3's exercises (with metadata like `source`, `chunk_index`) and load them
into a Chroma collection. Run 3 queries and, for each, report: the retrieved
chunk, its metadata, and whether metadata filtering (`where=...`) would help
or hurt for that particular query.
